# What generated this data?

Runs after `05_report_model.ipynb`. Reads `../data/clean.csv`.

Everything so far has treated the dataset as if it recorded runners. It does not — it is synthetic,
and the useful question is what process produced it. That is answerable from the output alone, and
answering it explains several earlier results at once.

Four checks: is the outcome additive, is the noise Gaussian, which inputs actually enter, and are
the inputs independent of each other.

Wording throughout: the evidence is *consistent with* a generating process of a particular shape.
None of this recovers the actual code.

In [ ]:
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

RS = 0
df = pd.read_csv('../data/clean.csv')
d = df[df.dnf == 0]

# duplicates removed so they cannot mask each other in the importance step:
# rest_days = 7 - runs_per_week, missed_workout_pct mirrors adherence, vo2_max mirrors resting HR
SPOILERS = ['actual_finish_time_minutes', 'target_finish_time_minutes', 'goal_gap', 'dnf',
            'medal', 'medal_outcome', 'runner_id', 'personal_best_minutes']
DUPES = ['rest_days_per_week', 'missed_workout_pct', 'vo2_max']
IN = [c for c in df.select_dtypes('number').columns
      if c not in SPOILERS + DUPES and not c.endswith('_was_missing')]

X, y = d[IN].values, d.actual_finish_time_minutes.values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=RS)
len(IN)

## Check 1: is the outcome additive?

Fit two gradient-boosted models. One is free to use interactions between variables; the other is
constrained to a sum of per-variable terms and nothing else. If the generator built finish time by
adding contributions together, the constraint costs nothing.

In [ ]:
full = HistGradientBoostingRegressor(max_iter=300, random_state=RS).fit(Xtr, ytr)
add = HistGradientBoostingRegressor(max_iter=300, random_state=RS,
                                    interaction_cst=[[i] for i in range(X.shape[1])]).fit(Xtr, ytr)

print(f'unconstrained (interactions allowed) R2 = {full.score(Xte, yte):.4f}')
print(f'additive only                        R2 = {add.score(Xte, yte):.4f}')
print(f'gain from allowing interactions        = {full.score(Xte, yte) - add.score(Xte, yte):+.4f}')

The difference is four ten-thousandths of R^2, and it flips sign with the random seed. With 19,500
held-out rows and unrestricted interaction search, that is indistinguishable from zero: a sum of
separate per-variable terms already captures everything the outcome contains.

This is the structural explanation for `04`. We asked whether the drivers of finish time differ by
profile, and the answer was no once the model was specified properly. An additive generator has no
interactions to find — not between profiles, not between anything.

## Check 2: what is left over

If the process is `finish_time = f(inputs) + noise`, then the residuals of a model that has
recovered `f` are the noise term itself, and its shape is readable.

In [ ]:
resid = yte - add.predict(Xte)

print(f'residual SD       {resid.std():.2f} minutes')
print(f'skew              {stats.skew(resid):+.3f}   (0 = symmetric)')
print(f'excess kurtosis   {stats.kurtosis(resid):+.3f}   (0 = normal tails)')

# constant spread, or does it grow with the prediction?
spread = (pd.Series(resid)
            .groupby(pd.qcut(add.predict(Xte), 5, labels=['lowest', '2', '3', '4', 'highest']),
                     observed=True).std().round(2))
spread.rename('residual SD by predicted quintile')

Symmetric, normal-tailed, and the spread is flat across the whole prediction range. That is an
additive Gaussian error with a fixed standard deviation of about 21 minutes — not a
multiplicative or heteroskedastic one.

It also puts a ceiling on the whole exercise. Roughly 21 minutes of every runner's time is drawn
from a random number generator, so no model of this dataset can do much better than the 18-minute
average error in `05`. We are close to the ceiling already.

## Check 3: which inputs the process actually used

Permutation importance on the additive model: shuffle one column, see how much R^2 falls.

In [ ]:
imp = permutation_importance(add, Xte, yte, n_repeats=3, random_state=RS, scoring='r2')
importance = pd.Series(imp.importances_mean, index=IN).sort_values(ascending=False)

print(f'{(importance <= 0.0005).sum()} of {len(IN)} columns fall below 0.0005')
importance[importance > 0.0005].round(4).rename('R2 lost when shuffled')

Running experience is worth about thirteen times everything else put together. Injury count and
resting heart rate contribute a little. Below those, nothing.

This is the same fact `03` reported as a correlation table — most columns are flat against finish
time — restated as a statement about the process: those columns were never in the equation. The
adherence and wellness block is not weakly informative, it is decorative.

## Check 4: are the inputs related to each other?

A generator that draws each column independently leaves a correlation matrix that is empty apart
from whatever was deliberately engineered.

In [ ]:
ALL_IN = [c for c in df.select_dtypes('number').columns
          if c not in SPOILERS and not c.endswith('_was_missing')]
C = df[ALL_IN].corr().abs().values.copy()
np.fill_diagonal(C, 0)
C = pd.DataFrame(C, index=ALL_IN, columns=ALL_IN)

pairs = C.values[np.triu_indices(len(ALL_IN), 1)]
print(f'{len(pairs)} input pairs, median |r| = {np.median(pairs):.4f}')
for t in (0.1, 0.2, 0.5):
    print(f'  |r| above {t}: {(pairs > t).sum()} pairs ({(pairs > t).mean():.0%})')
C.unstack().sort_values(ascending=False)[::2].head(8).round(3).rename('|r|')

Median 0.004, and most pairs sit near zero — but this is not a clean identity matrix. About one
pair in nine exceeds 0.2, and those form blocks rather than scattering randomly: `rest_days` is
`7 - runs_per_week`, adherence and missed workouts are one quantity from opposite ends, VO2 max
and resting heart rate are two readings of one fitness variable, and a motivation block has one
latent trait driving early-morning runs, club attendance and training in bad weather.

So the shape is a handful of latent variables with several observed columns derived from each,
and everything else drawn on its own.

This answers the VO2 max question directly. Training volume cannot explain VO2 max in this dataset
because VO2 max was not drawn from training volume. It was drawn on its own.

## The spoilers, confirmed

`02` kept `target_finish_time_minutes` and `personal_best_minutes` out of anything predicting
finish time, on the grounds that they leak. They leak in a specific direction.

In [ ]:
gap = d.actual_finish_time_minutes - d.target_finish_time_minutes
print(f'corr(target, actual) = {np.corrcoef(d.target_finish_time_minutes, d.actual_finish_time_minutes)[0, 1]:.3f}')
gap.describe().round(2).rename('actual minus target (minutes)')

The gap is a tight positive bundle between 11 and 60 minutes with no runner ever beating their
target. The target was not a goal the runner set and then missed — it was produced by subtracting a
random amount from the finish time after the fact. It is downstream of the outcome, not upstream.

The instinct in `02` was right; this is the measurement of it.

## Bounds

One last fingerprint, visible in the marginal distributions.

In [ ]:
bounds = pd.DataFrame({
    'min': df[['actual_finish_time_minutes', 'vo2_max', 'resting_heart_rate_bpm', 'bmi',
               'sleep_hours_avg', 'training_adherence_pct', 'weekly_mileage_miles',
               'long_run_distance_km']].min(),
    'max': df[['actual_finish_time_minutes', 'vo2_max', 'resting_heart_rate_bpm', 'bmi',
               'sleep_hours_avg', 'training_adherence_pct', 'weekly_mileage_miles',
               'long_run_distance_km']].max()})
bounds['at_min'] = [(df[c] == df[c].min()).mean().round(3) for c in bounds.index]
bounds.round(2)

Round numbers at both ends of almost every column: 35 to 75, 45 to 80, 18 to 32, 5 to 10, 40 to
100. Values were drawn and then clipped into a plausible range. Where the draw ran well past the
bound, the clip piles a large share of rows onto the boundary itself — which is exactly the 81% at
15.0 km that made `03` exclude `long_run_distance_km`, now explained rather than merely observed.

## Figure

In [ ]:
plt.rcParams.update({'figure.dpi': 150, 'font.size': 8, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.titlesize': 9, 'axes.titleweight': 'bold',
                     'axes.labelsize': 8, 'legend.frameon': False})
INK, ACC, GREY = '#2b2b2b', '#c0392b', '#9aa0a6'
os.makedirs('../figures', exist_ok=True)

fig, ax = plt.subplots(2, 2, figsize=(11, 7))

# A: additive vs unconstrained
a = ax[0, 0]
vals = [add.score(Xte, yte), full.score(Xte, yte)]
a.bar(['additive only', 'interactions allowed'], vals, color=[INK, GREY], width=0.5)
for i, v in enumerate(vals):
    a.text(i, v + 0.008, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')
a.set_ylim(0, 0.72); a.set_ylabel('test R²')
a.set_title('Interactions add nothing')

# B: the noise
a = ax[0, 1]
a.hist(resid, bins=70, density=True, color=GREY)
xs = np.linspace(resid.min(), resid.max(), 300)
a.plot(xs, stats.norm.pdf(xs, resid.mean(), resid.std()), color=ACC, lw=1.6,
       label=f'normal, SD {resid.std():.1f}')
a.set_title('What is left over is Gaussian')
a.set_xlabel('residual (minutes)'); a.set_ylabel('density'); a.legend(fontsize=7)

# C: importance
a = ax[1, 0]
top = importance.head(8)[::-1]
a.barh(top.index, top.values, color=[ACC if v > 0.01 else GREY for v in top.values])
a.set_xscale('log'); a.set_xlim(1e-4, 3)
a.set_title('Three variables, then nothing')
a.set_xlabel('R² lost when the column is shuffled (log scale)')
a.tick_params(labelsize=6.5)

# D: independence
a = ax[1, 1]
im = a.imshow(C.values, cmap='Reds', vmin=0, vmax=1)
a.set_xticks([]); a.set_yticks([])
a.set_title(f'Mostly independent draws (median |r| = {np.median(pairs):.3f})')
a.set_xlabel(f'{len(ALL_IN)} input columns; the bright blocks are derived groups')
fig.colorbar(im, ax=a, shrink=0.75, pad=0.02).set_label('|correlation|', fontsize=7)

fig.suptitle('The data generating process, read off its output', fontsize=11, fontweight='bold',
             x=0.02, ha='left', y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig('../figures/fig5_generator.png', bbox_inches='tight')

## What this is worth to the report

The process is consistent with: draw each column independently from a bounded distribution, clip
it, derive a few columns from others (rest days, missed workouts, VO2 max, a motivation block),
compute finish time as a sum of separate contributions dominated by running experience, add
Gaussian noise of about 21 minutes, then derive the target time by subtracting a random gap from
the result.

That single description accounts for four findings that were separate puzzles until now:

- 26 of 37 columns are flat against finish time — they were never in the equation.
- The drivers do not differ by profile — an additive process has no interactions to differ.
- Training volume does not explain VO2 max — VO2 max belongs to a different derived block.
- The volume columns pile up on their minimum — that is the clip, not a population of low-mileage runners.

The limit of the claim: this is the shape the output is consistent with, not the code. A different
generator could produce the same fingerprints. What can be said without hedging is the negative —
this dataset cannot support conclusions about how training affects marathon performance, because
the relationships in it were assigned rather than observed.